Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../results', exist_ok=True)
print("Library siap!")

 Load Dataset Bersih

In [ ]:
df = pd.read_csv('../data/dataset_bersih.csv')
print("Shape:", df.shape)
df.head()

 Persiapan Fitur & Target

In [ ]:
# Fitur input (X) dan target (y)
fitur = ['tahun', 'bulan', 'minggu', 'kuartal', 'kategori_encoded']
target = 'jumlah_order'

X = df[fitur]
y = df[target]

# Split data 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Fungsi Evaluasi

In [ ]:
def evaluasi(nama_model, y_test, y_pred):
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    print(f"\n{'='*35}")
    print(f"  {nama_model}")
    print(f"{'='*35}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  MAPE : {mape:.2f}%")
    
    return {'Model': nama_model, 'RMSE': round(rmse,4), 
            'MAE': round(mae,4), 'MAPE': round(mape,2)}

Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

hasil_lr = evaluasi("Linear Regression", y_test, y_pred_lr)

Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

hasil_rf = evaluasi("Random Forest", y_test, y_pred_rf)

XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

hasil_xgb = evaluasi("XGBoost", y_test, y_pred_xgb)

LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

# Scaling dulu untuk LSTM
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1))

# Reshape untuk LSTM (samples, timesteps, features)
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_test_lstm  = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

# Arsitektur LSTM
model_lstm = Sequential([
    LSTM(64, input_shape=(1, X_train_scaled.shape[1]), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train_lstm, y_train_scaled, 
               epochs=50, batch_size=32, 
               validation_split=0.1, verbose=0)

# Prediksi & inverse scaling
y_pred_lstm_scaled = model_lstm.predict(X_test_lstm)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled).flatten()

hasil_lstm = evaluasi("LSTM", y_test, y_pred_lstm)

Tabel Perbandingan Semua Model

In [ ]:
# Gabungkan semua hasil
semua_hasil = pd.DataFrame([hasil_lr, hasil_rf, hasil_xgb, hasil_lstm])
semua_hasil = semua_hasil.sort_values('RMSE')

print("\nPERBANDINGAN MODEL")
print(semua_hasil.to_string(index=False))

# Simpan ke CSV
semua_hasil.to_csv('../results/perbandingan_model.csv', index=False)

# Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrik = ['RMSE', 'MAE', 'MAPE']
warna  = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for i, m in enumerate(metrik):
    axes[i].bar(semua_hasil['Model'], semua_hasil[m], color=warna)
    axes[i].set_title(f'Perbandingan {m}')
    axes[i].set_ylabel(m)
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Perbandingan Performa Model Forecasting', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../hasil/perbandingan_model.png', dpi=150)
plt.show()

print("Hasil tersimpan di folder hasil/")